In [5]:
import pandas as pd
import numpy as np
import os
import joblib

silver_path = "../data/silver/"
gold_path = "../data/gold/"
models_path = "../models/"
os.makedirs(gold_path, exist_ok=True)

# 1. Cargar el modelo ganador y los mapeos
model = joblib.load(f"{models_path}modelo_next_best_product.pkl")
print("Modelo predictivo cargado correctamente.")

# 2. Cargar datos de clientes activos para evaluar
df_cli = pd.read_parquet(f"{silver_path}clientes.parquet")
df_est = pd.read_parquet(f"{silver_path}cliente_estado_mensual.parquet")
df_prod_cat = pd.read_parquet(f"{silver_path}productos.parquet")

# Combinar para tener el set de scoring
df_scoring = pd.merge(df_est, df_cli, on='id_cliente', how='inner')
print(f"Los clientes listos para scoring comercial son: {df_scoring['id_cliente'].nunique()}")

Modelo predictivo cargado correctamente.
Los clientes listos para scoring comercial son: 2000


In [6]:
# BLOQUE INTEGRADO: GENERACIÓN DE CANDIDATOS Y SCORING
print("Construyendo matriz de recomendación...")

# 1. Crear combinaciones de todos los clientes con todos los productos (Cross Join)
df_scoring['key'] = 1
df_prod_cat['key'] = 1
df_candidatos = pd.merge(df_scoring, df_prod_cat, on='key').drop(columns=['key'])

# 2. Aplicar One-Hot Encoding automático para transformar texto a columnas numéricas
X_scoring = pd.get_dummies(df_candidatos, drop_first=True)

# 3. ALINEACIÓN MATEMÁTICA: Recuperar las columnas exactas que exige el modelo entrenado
columnas_modelo = model.feature_names_in_
X_scoring_final = X_scoring.reindex(columns=columnas_modelo, fill_value=0)
X_scoring_final = X_scoring_final.fillna(0)

print(f"Matriz alineada con éxito. Columnas para el modelo: {X_scoring_final.shape[1]}")


# PREDICCIONES DE MACHINE LEARNING 

print("\nEjecutando predicciones de Machine Learning...")

# 4. Calcular probabilidades de éxito con el Random Forest
df_candidatos['probabilidad_adquisicion'] = model.predict_proba(X_scoring_final)[:, 1]

# 5. Crear el ranking por cliente (de mayor a menor probabilidad) [cite: 22]
df_candidatos['ranking_producto'] = df_candidatos.groupby('id_cliente')['probabilidad_adquisicion'].rank(ascending=False, method='first').astype(int)

# 6. Filtrar las columnas requeridas por el negocio [cite: 213]
df_predicciones_final = df_candidatos[[
    'id_cliente', 'fecha_corte', 'id_producto', 'nombre_producto', 
    'categoria_producto', 'probabilidad_adquisicion', 'ranking_producto'
]].rename(columns={'id_producto': 'id_producto_candidato'})

# Guardar archivo de predicciones en la capa Gold [cite: 211]
df_predicciones_final.to_parquet(f"{gold_path}predicciones_next_best_product.parquet", index=False)
print(" El archivo 'predicciones_next_best_product.parquet' fue generado y guardado en la capa Gold con éxito.")

Construyendo matriz de recomendación...
Matriz alineada con éxito. Columnas para el modelo: 16

Ejecutando predicciones de Machine Learning...
 El archivo 'predicciones_next_best_product.parquet' fue generado y guardado en la capa Gold con éxito.


In [7]:
print("Filtrando la mejor recomendación por cliente...")

# Quedarnos solo con el Top 1 del ranking
df_recomendaciones = df_predicciones_final[df_predicciones_final['ranking_producto'] == 1].copy()

# Adecuar nombres de columnas según la estructura final solicitada por el negocio
df_recomendaciones = df_recomendaciones.rename(columns={
    'fecha_corte': 'fecha_recomendacion',
    'nombre_producto': 'producto_recomendado',
    'probabilidad_adquisicion': 'probabilidad',
    'ranking_producto': 'posicion_ranking'
})

# Seleccionar campos requeridos
df_recomendaciones_entrega = df_recomendaciones[[
    'id_cliente', 'fecha_recomendacion', 'producto_recomendado', 
    'categoria_producto', 'probabilidad', 'posicion_ranking'
]]

# Guardar reporte de recomendaciones en la capa Gold
df_recomendaciones_entrega.to_parquet(f"{gold_path}recomendaciones_next_best_product.parquet", index=False)

print("\n¡ EL PROYECTO FUE COMPLETADO EXITOSAMENTE!")
print(f" El archivo final 'recomendaciones_next_best_product.parquet' se guardo con {len(df_recomendaciones_entrega)} clientes listos.")
print("\nResultado final:")
display(df_recomendaciones_entrega.head(5))

Filtrando la mejor recomendación por cliente...

¡ EL PROYECTO FUE COMPLETADO EXITOSAMENTE!
 El archivo final 'recomendaciones_next_best_product.parquet' se guardo con 2000 clientes listos.

Resultado final:


,id_cliente,fecha_recomendacion,producto_recomendado,categoria_producto,probabilidad,posicion_ranking
0,1375586,2015-01-28,Cuenta de ahorro,Ahorro,0.18,1
16,1050611,2015-01-28,Cuenta de ahorro,Ahorro,0.11,1
32,1050612,2015-01-28,Cuenta de ahorro,Ahorro,0.08,1
48,1050613,2015-01-28,Cuenta de ahorro,Ahorro,0.08,1
64,1050614,2015-01-28,Cuenta de ahorro,Ahorro,0.02,1
